# 04. 정상상담 vs 보이스피싱 모델 가상 시나리오 검증

기존에 학습·저장한 모델을 변경하지 않고, 학습에 사용하지 않은 가상 시나리오를 입력하여 오탐과 미탐을 확인합니다.

## 검증 질문

1. 명확한 보이스피싱을 제대로 탐지하는가?
2. 직접적인 위험 단어가 적은 은밀한 보이스피싱도 탐지하는가?
3. 계좌·송금·인증번호가 등장하는 정상 금융상담을 보이스피싱으로 오탐하지 않는가?
4. 학습 범위 밖의 일반 대화를 억지로 보이스피싱으로 분류하지 않는가?
5. 판단하기 어려운 짧거나 모호한 문장에서 점수가 어떻게 나타나는가?

가상 시나리오는 도전용 점검 자료이며 실제 외부 데이터 성능을 대신하지 않습니다.


In [ ]:
# 0. 라이브러리와 한글 폰트 설치
!pip -q install pandas scikit-learn seaborn matplotlib joblib openpyxl
!apt-get -qq update
!apt-get -qq install -y fonts-nanum


In [ ]:
# 1. 라이브러리·Google Drive·한글 폰트
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from IPython.display import display
import json, re, unicodedata, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
font_paths=sorted(Path('/usr/share/fonts/truetype/nanum').glob('*.ttf'))
assert font_paths,'나눔글꼴 설치 셀을 먼저 실행하세요.'
for font_path in font_paths: font_manager.fontManager.addfont(str(font_path))
preferred=next((p for p in font_paths if p.name=='NanumGothic.ttf'),font_paths[0])
korean_font=font_manager.FontProperties(fname=str(preferred)).get_name()
mpl.rcParams['font.family']=korean_font
mpl.rcParams['font.sans-serif']=[korean_font]
mpl.rcParams['axes.unicode_minus']=False
print('한글 폰트:',korean_font)


## 1. 경로와 고정 모델 불러오기


In [ ]:
# 2. 모델 후보 경로와 결과 경로
PROJECT_ROOT=Path('/content/drive/MyDrive/보이스피싱_분석')
MODEL_CANDIDATES=[
 PROJECT_ROOT/'머신러닝_분석결과_v5_통합'/'03_텍스트ML'/'03_모델'/'fraud_detection_best_model.joblib',
 PROJECT_ROOT/'머신러닝_분석결과_v4'/'03_모델'/'fraud_detection_best_model.joblib',
 PROJECT_ROOT/'머신러닝_분석결과_v4_사건단위'/'03_모델'/'fraud_detection_best_model.joblib',
]
MODEL_PATH=next((p for p in MODEL_CANDIDATES if p.exists()),None)
if MODEL_PATH is None:
    discovered=list(PROJECT_ROOT.rglob('fraud_detection_best_model.joblib'))
    MODEL_PATH=max(discovered,key=lambda p:p.stat().st_mtime) if discovered else None
assert MODEL_PATH is not None and MODEL_PATH.exists(), 'fraud_detection_best_model.joblib을 찾지 못했습니다.'

OUTPUT_ROOT=PROJECT_ROOT/'머신러닝_분석결과_v5_4_가상시나리오검증'
SCENARIO_ROOT=OUTPUT_ROOT/'01_시나리오'
RESULT_ROOT=OUTPUT_ROOT/'02_예측결과'
FIGURE_ROOT=OUTPUT_ROOT/'03_그래프'
ERROR_ROOT=OUTPUT_ROOT/'04_오류분석'
REPORT_ROOT=OUTPUT_ROOT/'05_보고서'
for folder in [SCENARIO_ROOT,RESULT_ROOT,FIGURE_ROOT,ERROR_ROOT,REPORT_ROOT]: folder.mkdir(parents=True,exist_ok=True)

# 신뢰할 수 있는 본 프로젝트에서 직접 저장한 joblib만 불러옵니다.
model=joblib.load(MODEL_PATH)
assert hasattr(model,'predict'),'예측 가능한 모델이 아닙니다.'
print('고정 모델:',MODEL_PATH)
print('모델 클래스:',getattr(model,'classes_','확인 불가'))


## 2. 가상 시나리오 데이터 생성

총 72개를 생성합니다. 정답이 있는 네 집단은 각 16개, 모호한 판단유보 집단은 8개입니다.


In [ ]:
# 3. 가상 시나리오 원문
CLEAR_FRAUD = [
"서울중앙지검 수사관입니다. 고객님 명의 계좌가 범죄에 연루됐으니 누구에게도 알리지 말고 안전계좌로 전액 이체하세요.",
"경찰청 금융범죄팀입니다. 체포영장이 발부될 수 있으니 통화를 끊지 말고 지금 가까운 은행에서 현금을 인출하세요.",
"금융감독원 직원입니다. 계좌가 불법 자금에 사용되어 검사해야 하니 잔액을 지정 계좌로 송금해 주세요.",
"저금리 대환대출 승인을 위해 기존 대출금을 먼저 상환해야 합니다. 오늘 안으로 안내한 계좌에 입금하세요.",
"대출 실행 전 보증료가 필요합니다. 심사가 취소되기 전에 지금 수수료를 보내셔야 합니다.",
"카드가 해외에서 발급됐습니다. 피해를 막으려면 카드번호와 비밀번호, 인증번호를 바로 알려주세요.",
"아드님이 사고를 내서 합의금이 급합니다. 다른 가족에게 연락하지 말고 지금 현금을 준비해 전달하세요.",
"법원 집행관입니다. 미납 과태료 때문에 계좌가 압류될 예정이니 오늘 중 지정 계좌로 납부하세요.",
"은행 보안팀입니다. 악성 거래를 차단하려면 보내드린 링크를 누르고 원격지원 앱을 설치해야 합니다.",
"수사 보안을 위해 전화는 끊으면 안 됩니다. 주민번호와 계좌번호를 확인한 뒤 자금을 보호계좌로 옮기세요.",
"검사실에서 연락드렸습니다. 사건번호가 등록됐고 출석 전 자금 검수가 필요하니 전액 이체하십시오.",
"정부 지원 대출 대상입니다. 한도 확보를 위해 선입금이 필요하니 지금 백만 원을 보내주세요.",
"택배에 불법 물품이 발견됐습니다. 처벌을 피하려면 안내 주소로 접속해 본인정보와 카드번호를 입력하세요.",
"저축은행 채권팀입니다. 연체 등록을 막으려면 기존 채무를 오늘 안에 상환해야 합니다.",
"금융사고 조사 중이라 은행 직원에게도 말하면 안 됩니다. 현금을 찾아 지정한 직원에게 전달하세요.",
"검찰 피해자 보호 절차입니다. 통화 녹취 중이니 지시에 따라 보안카드 번호와 OTP를 말씀하세요.",
]

SUBTLE_FRAUD = [
"고객님은 우대 전환 대상입니다. 한도를 유지하려면 기존 채무를 정리하는 절차가 먼저 필요해 안내 계좌를 보내드리겠습니다.",
"본인 확인 단계가 남아 있습니다. 처리 지연을 막기 위해 다른 상담 창구와 연락하지 마시고 제가 안내하는 순서대로 진행해 주세요.",
"거래 보호 절차상 잔액을 잠시 분리해 두어야 합니다. 확인이 끝나면 원래 상태로 복원됩니다.",
"심사 승인 번호가 발급됐지만 오늘 처리하지 않으면 자동 취소됩니다. 준비 가능한 금액부터 접수하겠습니다.",
"고객님의 거래가 확인 대상에 포함됐습니다. 외부에 내용이 전달되면 처리가 중단되니 이 통화에서만 확인하겠습니다.",
"기존 상품보다 낮은 조건으로 변경할 수 있습니다. 우선 현재 이용 중인 금액을 정리한 내역이 필요합니다.",
"보안상 일반 창구에서는 조회되지 않는 절차입니다. 제가 연결해 드리는 담당자의 안내만 따라주세요.",
"확인용 금액은 절차가 끝나면 반환됩니다. 먼저 가능한 범위를 말씀해 주시면 방법을 안내하겠습니다.",
"계정 보호를 위해 임시 인증 과정이 필요합니다. 문자로 도착한 숫자를 읽어주시면 바로 마무리됩니다.",
"명의 사용 여부를 확인하고 있습니다. 거래하시는 은행과 현재 잔액을 순서대로 말씀해 주세요.",
"지원 대상자로 최종 선정되셨습니다. 접수 상태를 유지하려면 오늘 안에 처리 기록이 있어야 합니다.",
"담당 부서에서 직접 관리하는 건이라 다른 직원은 내용을 모를 수 있습니다. 연락은 이 번호로만 받아주세요.",
"환급 절차가 승인됐습니다. 반환 계좌 확인을 위해 먼저 소액 인증을 진행하겠습니다.",
"고객 보호 조치가 시작됐습니다. 전화를 종료하면 처음부터 다시 해야 하니 연결 상태를 유지해 주세요.",
"서류상 본인 명의 거래로 확인됩니다. 사실관계를 정리하기 위해 최근 거래내역 사진을 보내주세요.",
"한도 조정이 가능한 상태입니다. 시스템 반영 전에 필요한 절차를 지금 함께 처리하겠습니다.",
]

HARD_NORMAL = [
"고객님이 신청하신 계좌이체 한도 변경이 완료됐습니다. 당사 직원은 비밀번호나 인증번호를 요구하지 않습니다.",
"대출 상환 예정일을 안내드립니다. 납부는 공식 앱이나 영업점에서만 가능하며 개인 계좌로 송금을 요청하지 않습니다.",
"카드 해외 결제 여부를 확인하는 상담입니다. 카드번호 전체와 비밀번호는 말씀하지 않으셔도 됩니다.",
"보이스피싱 예방을 위해 검찰이나 경찰을 사칭하며 안전계좌 이체를 요구하면 즉시 통화를 종료해 주세요.",
"고객님 계좌의 입금 내역을 확인했습니다. 추가 확인은 공식 홈페이지나 가까운 은행에서 진행할 수 있습니다.",
"보험료 자동이체 계좌를 변경하려면 본인이 직접 앱에서 인증하거나 영업점에 방문하셔야 합니다.",
"대출금리는 심사 후 확정되며 승인 전에 수수료나 보증금을 먼저 입금하도록 요구하지 않습니다.",
"분실 카드 신고가 접수됐습니다. 새 카드 발급 과정에서 상담원이 OTP나 비밀번호를 묻지 않습니다.",
"송금 취소 가능 여부를 확인해 드리겠습니다. 거래번호만 확인하고 계좌 비밀번호는 받지 않습니다.",
"금융감독원은 자금 검사를 이유로 개인에게 현금 인출이나 계좌이체를 요구하지 않습니다.",
"주민번호는 입력하지 마시고 생년월일 여섯 자리만 자동응답 시스템에 직접 입력해 주세요.",
"적금 만기금은 등록된 본인 계좌로 자동 입금됩니다. 별도의 안전계좌 이체 절차는 없습니다.",
"현재 상담은 상품 설명이며 즉시 결정하지 않으셔도 됩니다. 내용을 확인한 뒤 공식 번호로 다시 연락하셔도 됩니다.",
"거래내역 발급 수수료는 영업점 창구에서 고지되며 직원 개인 계좌로 납부하지 않습니다.",
"앱 설치가 필요하다면 공식 앱스토어에서 은행 이름을 직접 검색해 주세요. 문자 링크는 사용하지 마십시오.",
"의심되는 전화가 있었다면 송금하지 말고 경찰과 해당 은행의 공식 고객센터에 신고해 주세요.",
]

GENERAL_NORMAL = [
"안녕하세요. 내일 오후 세 시 병원 예약을 확인하려고 전화드렸습니다.",
"주문하신 상품이 오늘 배송될 예정이며 부재 시 경비실에 맡겨드리겠습니다.",
"회의 일정이 금요일 오전으로 변경됐으니 참석 가능 여부를 알려주세요.",
"도서관에서 대출한 책의 반납 예정일이 다음 주 월요일입니다.",
"차량 정비가 완료되어 영업시간 안에 방문하시면 찾아가실 수 있습니다.",
"이번 주 수업 과제는 강의실 게시판에 올렸으니 확인해 주세요.",
"예약하신 식당 좌석이 준비됐으며 인원 변경이 있으면 미리 연락해 주세요.",
"아파트 수도 점검이 내일 진행되니 오전 중 잠시 단수가 있을 예정입니다.",
"면접 결과는 다음 주에 이메일로 안내드릴 예정입니다.",
"택배 주소에 동 번호가 빠져 있어 정확한 배송지만 확인하려고 연락드렸습니다.",
"수리 기사 방문 시간은 오후 두 시에서 네 시 사이입니다.",
"동아리 모임 장소가 학교 앞 카페로 변경됐습니다.",
"검진 결과 상담 예약을 위해 가능한 날짜를 확인하고 있습니다.",
"학부모 상담 일정표를 문자로 보내드렸으니 희망 시간을 선택해 주세요.",
"온라인으로 주문한 의자의 색상을 변경할 수 있는지 문의드립니다.",
"주말 행사에 참여하실 분은 오늘까지 이름만 남겨주세요.",
]

AMBIGUOUS_REVIEW = [
"지금 확인이 필요하니 안내하는 대로 해주세요.",
"본인 명의가 맞는지만 확인하겠습니다.",
"처리가 끝나면 다시 돌려드릴 예정입니다.",
"오늘 안에 완료하지 않으면 접수가 취소됩니다.",
"계좌 관련해서 잠깐 확인할 내용이 있습니다.",
"다른 곳에는 연락하지 말고 기다려 주세요.",
"문자로 받은 번호를 확인해 주시겠어요.",
"담당자 연결 후 필요한 절차를 안내하겠습니다.",
]

scenario_specs=[
 ('CLEAR_FRAUD','명확한 보이스피싱','VOICE_PHISHING',CLEAR_FRAUD,'사칭·위협·금전·정보 요구가 명시됨'),
 ('SUBTLE_FRAUD','은밀한 보이스피싱','VOICE_PHISHING',SUBTLE_FRAUD,'직접적 범죄 단어를 줄인 사회공학적 요구'),
 ('HARD_NORMAL','위험 금융용어 정상상담','LEGITIMATE_FINANCIAL_CALL',HARD_NORMAL,'위험 단어를 포함하지만 실제로는 예방·정상 절차 안내'),
 ('GENERAL_NORMAL','일반 정상대화','LEGITIMATE_FINANCIAL_CALL',GENERAL_NORMAL,'금융상담 학습 범위 밖의 일상 대화'),
 ('AMBIGUOUS_REVIEW','판단유보','REVIEW',AMBIGUOUS_REVIEW,'짧거나 문맥이 부족해 정답 확정이 어려움'),
]
rows=[]
for code_name,korean_name,label,texts,rationale in scenario_specs:
    for number,text in enumerate(texts,1):
        rows.append({'scenario_id':f'{code_name}_{number:02d}','scenario_group':code_name,
          '시나리오구분':korean_name,'expected_label':label,'scenario_text':text,
          '설계의도':rationale,'scenario_source':'SYNTHETIC_MANUAL_V1'})
scenario_df=pd.DataFrame(rows)
display(scenario_df.groupby(['시나리오구분','expected_label']).size().reset_index(name='건수'))
scenario_df.to_csv(SCENARIO_ROOT/'가상시나리오_72개.csv',index=False,encoding='utf-8-sig')
print('가상 시나리오:',len(scenario_df),'개')


## 3. 사용자가 수정한 시나리오로 교체하기

기본 72개 대신 직접 수정한 CSV를 사용할 경우 아래 설정을 변경합니다. 필수 컬럼은 `scenario_id`, `scenario_group`, `expected_label`, `scenario_text`입니다.


In [ ]:
# 4. 선택적으로 사용자 시나리오 CSV 사용
USE_CUSTOM_SCENARIOS=False
CUSTOM_SCENARIO_PATH=SCENARIO_ROOT/'가상시나리오_사용자수정.csv'
if USE_CUSTOM_SCENARIOS:
    assert CUSTOM_SCENARIO_PATH.exists(),CUSTOM_SCENARIO_PATH
    scenario_df=pd.read_csv(CUSTOM_SCENARIO_PATH,encoding='utf-8-sig')
required={'scenario_id','scenario_group','expected_label','scenario_text'}
assert required.issubset(scenario_df.columns),f'필수 컬럼 누락: {required-set(scenario_df.columns)}'
assert scenario_df.scenario_id.is_unique,'scenario_id가 중복됩니다.'
print('평가 시나리오:',len(scenario_df),'개')


## 4. 모델 입력과 점수 계산


In [ ]:
# 5. 학습 노트북과 동일한 텍스트 정리
def clean_text(value):
    text=unicodedata.normalize('NFKC',str(value or '')).lower()
    text=re.sub(r'https?://\S+|www\.\S+',' ',text)
    text=re.sub(r'(?m)^\s*(tx|rx|화자\s*\d*|범인|피해자)\s*[:：]\s*',' ',text)
    text=re.sub(r'[*#xX]{2,}',' 마스킹 ',text)
    text=re.sub(r'\b\d{2,}\b',' 숫자 ',text)
    return re.sub(r'\s+',' ',text).strip()

def fraud_score(model,texts):
    classes=list(model.classes_)
    fraud_index=classes.index('VOICE_PHISHING')
    if hasattr(model,'predict_proba'):
        return model.predict_proba(texts)[:,fraud_index],'predict_proba'
    raw=np.asarray(model.decision_function(texts))
    if raw.ndim>1: raw=raw[:,fraud_index]
    elif fraud_index==0: raw=-raw
    # sigmoid는 순위 확인용 0~1 변환이며 보정된 확률이 아닙니다.
    return 1/(1+np.exp(-np.clip(raw,-30,30))),'decision_function_sigmoid'

scenario_df['clean_text']=scenario_df.scenario_text.fillna('').map(clean_text)
scenario_df['text_length']=scenario_df.clean_text.str.len()
scenario_df['predicted_label']=model.predict(scenario_df.clean_text)
scenario_df['fraud_score'],score_method=fraud_score(model,scenario_df.clean_text)
scenario_df['score_method']=score_method
scenario_df['정답여부']=np.where(scenario_df.expected_label.eq('REVIEW'),np.nan,
                               scenario_df.expected_label.eq(scenario_df.predicted_label))
scenario_df['예측한글']=scenario_df.predicted_label.map({
 'VOICE_PHISHING':'보이스피싱','LEGITIMATE_FINANCIAL_CALL':'정상 금융상담'}).fillna(scenario_df.predicted_label)
display(scenario_df.head())
scenario_df.to_csv(RESULT_ROOT/'가상시나리오_전체예측.csv',index=False,encoding='utf-8-sig')
print('점수 방식:',score_method)


## 5. 전체 및 시나리오별 평가


In [ ]:
# 6. 정답이 있는 64개만 정량평가에 사용
FRAUD='VOICE_PHISHING'; NORMAL='LEGITIMATE_FINANCIAL_CALL'
labeled=scenario_df[scenario_df.expected_label.isin([FRAUD,NORMAL])].copy()
p,r,f1,_=precision_recall_fscore_support(labeled.expected_label,labeled.predicted_label,
    average='binary',pos_label=FRAUD,zero_division=0)
tn,fp,fn,tp=confusion_matrix(labeled.expected_label,labeled.predicted_label,labels=[NORMAL,FRAUD]).ravel()
overall=pd.DataFrame([{
 '평가건수':len(labeled),'accuracy':accuracy_score(labeled.expected_label,labeled.predicted_label),
 'fraud_precision':p,'fraud_recall':r,'fraud_f1':f1,
 '정상_오탐률_FPR':fp/max(tn+fp,1),'사기_미탐률_FNR':fn/max(tp+fn,1),
 'TN':tn,'FP':fp,'FN':fn,'TP':tp,
}])
display(overall)
print(classification_report(labeled.expected_label,labeled.predicted_label,zero_division=0))
overall.to_csv(RESULT_ROOT/'전체_가상시나리오_성능.csv',index=False,encoding='utf-8-sig')

cm=np.array([[tn,fp],[fn,tp]])
plt.figure(figsize=(6,5)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
 xticklabels=['정상','보이스피싱'],yticklabels=['정상','보이스피싱'])
plt.xlabel('예측'); plt.ylabel('가상 정답'); plt.title('가상 시나리오 혼동행렬')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'01_가상시나리오_혼동행렬.png',dpi=170); plt.show()


In [ ]:
# 7. 시나리오 그룹별 포착률·오탐률
group_rows=[]
for group_name,g in scenario_df.groupby('scenario_group'):
    expected=g.expected_label.iloc[0]
    row={'scenario_group':group_name,'시나리오구분':g['시나리오구분'].iloc[0] if '시나리오구분' in g else group_name,
         '정답라벨':expected,'건수':len(g),'평균_보이스피싱점수':g.fraud_score.mean(),
         '중앙값_보이스피싱점수':g.fraud_score.median(),'보이스피싱예측률':g.predicted_label.eq(FRAUD).mean()}
    if expected==FRAUD: row['핵심지표명']='사기 포착률'; row['핵심지표']=g.predicted_label.eq(FRAUD).mean()
    elif expected==NORMAL: row['핵심지표명']='정상 오탐률'; row['핵심지표']=g.predicted_label.eq(FRAUD).mean()
    else: row['핵심지표명']='정량평가 제외'; row['핵심지표']=np.nan
    group_rows.append(row)
group_metrics=pd.DataFrame(group_rows)
display(group_metrics)
group_metrics.to_csv(RESULT_ROOT/'시나리오그룹별_성능.csv',index=False,encoding='utf-8-sig')

plt.figure(figsize=(12,6)); sns.boxplot(data=scenario_df,x='시나리오구분',y='fraud_score',order=[
 '명확한 보이스피싱','은밀한 보이스피싱','위험 금융용어 정상상담','일반 정상대화','판단유보'])
sns.stripplot(data=scenario_df,x='시나리오구분',y='fraud_score',color='black',alpha=.55,size=4)
plt.ylabel('보이스피싱 점수'); plt.xlabel(''); plt.xticks(rotation=18)
plt.title('가상 시나리오별 보이스피싱 점수 분포'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'02_시나리오별_점수분포.png',dpi=170); plt.show()


## 6. 오탐·미탐 및 출처 편향 점검


In [ ]:
# 8. 틀린 사례와 가장 자신 있게 틀린 사례
errors=labeled[~labeled['정답여부'].astype(bool)].copy()
errors['오류유형']=np.where(errors.expected_label.eq(FRAUD),'미탐_FN','오탐_FP')
errors['오류확신도']=np.where(errors.오류유형.eq('미탐_FN'),1-errors.fraud_score,errors.fraud_score)
errors=errors.sort_values('오류확신도',ascending=False)
display(errors[['scenario_id','시나리오구분','오류유형','fraud_score','scenario_text']])
errors.to_csv(ERROR_ROOT/'오탐_미탐_전체.csv',index=False,encoding='utf-8-sig')

hard_normal=scenario_df[scenario_df.scenario_group.eq('HARD_NORMAL')].sort_values('fraud_score',ascending=False)
subtle=scenario_df[scenario_df.scenario_group.eq('SUBTLE_FRAUD')].sort_values('fraud_score')
display(hard_normal[['scenario_id','fraud_score','predicted_label','scenario_text']].head(10))
display(subtle[['scenario_id','fraud_score','predicted_label','scenario_text']].head(10))
hard_normal.to_csv(ERROR_ROOT/'위험금융용어_정상상담_오탐점검.csv',index=False,encoding='utf-8-sig')
subtle.to_csv(ERROR_ROOT/'은밀한보이스피싱_미탐점검.csv',index=False,encoding='utf-8-sig')


In [ ]:
# 9. 문장 길이와 점수 관계: 길이를 이용하는지 보조 점검
length_corr=scenario_df[['text_length','fraud_score']].corr(method='spearman').iloc[0,1]
plt.figure(figsize=(9,6)); sns.scatterplot(data=scenario_df,x='text_length',y='fraud_score',hue='시나리오구분',s=70)
plt.title(f'문장 길이와 보이스피싱 점수: Spearman {length_corr:.3f}')
plt.xlabel('정제 텍스트 길이'); plt.ylabel('보이스피싱 점수'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'03_문장길이_점수관계.png',dpi=170); plt.show()
pd.DataFrame([{'Spearman_길이점수상관':length_corr}]).to_csv(
    RESULT_ROOT/'문장길이_점수상관.csv',index=False,encoding='utf-8-sig')


## 7. 사용자가 직접 추가 문장 시험


In [ ]:
# 10. 임의 문장을 추가로 입력하고 모델의 결과만 확인합니다.
NEW_SCENARIOS = [
    # "여기에 직접 만든 문장을 입력하세요",
]
if NEW_SCENARIOS:
    new_clean=[clean_text(x) for x in NEW_SCENARIOS]
    new_pred=model.predict(new_clean)
    new_score,_=fraud_score(model,new_clean)
    new_result=pd.DataFrame({'입력문장':NEW_SCENARIOS,'예측':new_pred,'보이스피싱점수':new_score})
    display(new_result)
    new_result.to_csv(RESULT_ROOT/'사용자추가문장_예측.csv',index=False,encoding='utf-8-sig')
else:
    print('NEW_SCENARIOS에 문장을 입력한 뒤 이 셀을 다시 실행하세요.')


## 8. 자동 보고서와 최종 점검


In [ ]:
# 11. 보고서 저장
metric_lookup=group_metrics.set_index('scenario_group')
clear_recall=float(metric_lookup.loc['CLEAR_FRAUD','핵심지표'])
subtle_recall=float(metric_lookup.loc['SUBTLE_FRAUD','핵심지표'])
hard_normal_fpr=float(metric_lookup.loc['HARD_NORMAL','핵심지표'])
general_normal_fpr=float(metric_lookup.loc['GENERAL_NORMAL','핵심지표'])
report=[
 '# 가상 시나리오 외부 모의검증 결과','', '## Summary','',
 f'- 사용 모델: {MODEL_PATH}',f'- 정답 포함 시나리오: {len(labeled)}개 / 판단유보: {scenario_df.expected_label.eq("REVIEW").sum()}개',
 f'- 전체 Accuracy: {overall.iloc[0].accuracy:.4f}',f'- 전체 보이스피싱 Recall: {overall.iloc[0].fraud_recall:.4f}',
 f'- 전체 정상 오탐률: {overall.iloc[0]["정상_오탐률_FPR"]:.4f}','',
 '## 핵심 도전 결과','',f'- 명확한 보이스피싱 포착률: {clear_recall:.2%}',
 f'- 은밀한 보이스피싱 포착률: {subtle_recall:.2%}',f'- 위험 금융용어 정상상담 오탐률: {hard_normal_fpr:.2%}',
 f'- 일반 정상대화 오탐률: {general_normal_fpr:.2%}','',
 '## 해석 주의','',
 '- 가상 시나리오는 모델 학습이나 선정에 사용하지 않은 도전용 자료입니다.',
 '- 사람이 작성한 문장이라 실제 통화의 ASR 오류, 말 끊김, 지역어와 잡음을 충분히 반영하지 못합니다.',
 '- 합성 시나리오 성능은 실제 외부 데이터 일반화 성능이 아닙니다.',
 '- 위험 금융용어 정상상담의 오탐과 은밀한 보이스피싱의 미탐을 우선 확인해야 합니다.',
 '- 모델을 수정한 뒤 같은 시나리오를 반복 선택 기준으로 쓰면 평가 자료에 과적합될 수 있으므로 버전을 고정해야 합니다.'
]
(REPORT_ROOT/'04_synthetic_scenario_validation_report.md').write_text('\n'.join(report),encoding='utf-8')
manifest={'version':'04_synthetic_v1','model_path':str(MODEL_PATH),'model_frozen_no_fit':True,
 'scenario_count':len(scenario_df),'labeled_count':len(labeled),'review_count':int(scenario_df.expected_label.eq('REVIEW').sum()),
 'score_method':score_method,'overall':overall.iloc[0].to_dict(),'group_metrics':group_metrics.to_dict(orient='records'),
 'synthetic_not_real_external_validation':True}
(REPORT_ROOT/'04_synthetic_run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

assert len(scenario_df)==72
assert len(labeled)==64
assert not scenario_df.scenario_id.duplicated().any()
assert scenario_df.fraud_score.between(0,1).all()
print('가상 시나리오 검증 정상 완료:',OUTPUT_ROOT)
